In [ ]:
GITHUB_REPO_URL = "https://github.com/24520152-ben/LLM-Generated-Multi-Level-Adversarial-Attacks-on-Vietnamese-Hate-Speech-Detection"
ATTACK_OPTIONS = {
    "checkpoint_dir": "/kaggle/input/datasets/hophamsailam/vihsd-baselines/baselines/visobert/checkpoint",
    "llm_checkpoint": "Qwen/Qwen3-30B-A3B-Instruct-2507",
    "llm_max_memory": ["11GiB", "11GiB"],
    "classifier_device": "cuda:1",
    "query_budgets": list(range(1, 26)),
    "beam_widths": [1, 3, 5],
    "modes": ["C", "W", "S", "MIX"],
    "sample_per_label": 10,
    "generation_budgets": list(range(5, 26, 5)),
    "output_dir": "/kaggle/working/artifacts/attacks/visobert",
}

In [ ]:
import json
import os
import subprocess
from pathlib import Path

kaggle_working = Path("/kaggle/working")
project_root = Path.cwd()
if kaggle_working.is_dir():
    project_root = kaggle_working / "ie403"
    if not project_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--", GITHUB_REPO_URL, str(project_root)],
            check=True,
        )
elif not (project_root / "src" / "vihsd_attack").is_dir():
    project_root = project_root.parent

source_dir = project_root / "src"
if not (source_dir / "vihsd_attack").is_dir():
    raise RuntimeError(f"Cannot find src/vihsd_attack in {project_root}.")
env = {
    **os.environ,
    "UV_PROJECT_ENVIRONMENT": str(project_root / ".venv-attacks"),
    "PYTHONPATH": str(source_dir),
}
if kaggle_working.is_dir() and not env.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

subprocess.run(
    [
        "uv",
        "sync",
        "--locked",
        "--python",
        "3.12",
        "--no-default-groups",
        "--group",
        "attacks",
    ],
    cwd=project_root,
    env=env,
    check=True,
)

attack_code = """
import json
import sys
from vihsd_attack.attacks.service import run_pilot
from vihsd_attack.attacks.config import AttackConfig

run_pilot(AttackConfig(**json.loads(sys.argv[1])))
"""
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "--no-default-groups",
        "--group",
        "attacks",
        "python",
        "-u",
        "-c",
        attack_code,
        json.dumps(ATTACK_OPTIONS, default=str),
    ],
    cwd=project_root,
    env=env,
    check=True,
)